# **Dataset cleaning and Preparation**

In [43]:
from datasets import load_dataset, Dataset
import pandas as pd

dataset = load_dataset("Nan-Do/instructional_code-search-net-python", split="train")
df = dataset.to_pandas()

In [44]:
def giving_info(df):
    print("Shape:")
    print(df.shape)

    print("\nData Types:")
    print(df.dtypes)

    print("\nColumns:")
    print(df.columns)

    print("\nDataset Information:")
    df.info()

    print("\nFirst 5 Rows:")
    print(df.head())

    print("\nduplicates values")
    print(df.duplicated().sum())

giving_info(df)

Shape:
(418545, 3)

Data Types:
INSTRUCTION    object
RESPONSE       object
SOURCE         object
dtype: object

Columns:
Index(['INSTRUCTION', 'RESPONSE', 'SOURCE'], dtype='object')

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418545 entries, 0 to 418544
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   INSTRUCTION  418545 non-null  object
 1   RESPONSE     418545 non-null  object
 2   SOURCE       418545 non-null  object
dtypes: object(3)
memory usage: 9.6+ MB

First 5 Rows:
                                         INSTRUCTION  \
0  Can you tell what is the following Python 3 fu...   
1  Can you generate a brief explanation for the f...   
2  How would you implement a function in Python 3...   
3  Implement a function in Python 3 to\nconvert a...   
4  Given the following Python 3 function, write t...   

                                            RESPONSE         SOURCE  
0  Train a k 

In [45]:
def find_missing_values(df):
    print("missing instruction", df['INSTRUCTION'].isna().sum())
    print("missing responses", df['RESPONSE'].isna().sum())

find_missing_values(df)

df = df.dropna(subset=["INSTRUCTION", "RESPONSE"])

df = df[
    (df["INSTRUCTION"].str.strip() != "") &
    (df["RESPONSE"].str.strip() != "")
]

missing instruction 0
missing responses 0


In [46]:
def empty_values(df):
    print("Empty instructions:", (df["INSTRUCTION"].fillna("").str.strip() == "").sum())
    print("Empty responses:", (df["RESPONSE"].fillna("").str.strip() == "").sum())

empty_values(df)

Empty instructions: 0
Empty responses: 0


In [47]:
def get_aggregation(df):
    print("aggregation of INSTRUCTION")
    print("Minimum:", df["INSTRUCTION"].str.len().min())
    print("Maximum:", df["INSTRUCTION"].str.len().max())
    print("Average:", df["INSTRUCTION"].str.len().mean())

    print("aggregation of RESPONSE")
    print("Minimum:", df["RESPONSE"].str.len().min())
    print("Maximum:", df["RESPONSE"].str.len().max())
    print("Average:", df["RESPONSE"].str.len().mean())

get_aggregation(df)

aggregation of INSTRUCTION
Minimum: 39
Maximum: 103724
Average: 534.8191281702087
aggregation of RESPONSE
Minimum: 7
Maximum: 55840
Average: 518.2902029650336


In [48]:
def length_of(df):
    print("Long Instructions:", (df["INSTRUCTION"].str.len() > 2000).sum())
    print("Long Responses:", (df["RESPONSE"].str.len() > 2000).sum())

length_of(df)

Long Instructions: 20610
Long Responses: 21350


In [49]:
def giving_info(df):
    print("Explanation:", df["INSTRUCTION"].str.contains("explain|explanation", case=False, na=False).sum())
    print("Code Generation:", df["INSTRUCTION"].str.contains("implement|write a function|generate code", case=False, na=False).sum())
    print("Documentation:", df["INSTRUCTION"].str.contains("documentation|document the code", case=False, na=False).sum())

giving_info(df)

Explanation: 111761
Code Generation: 79488
Documentation: 56695


In [50]:
def filter_explanations(df):
    explanation_df = df[df["INSTRUCTION"].str.contains("explain|explanation|what .* doing|what does .* do", case=False, na=False)]
    return explanation_df

explanation_df = filter_explanations(df)

print("Original examples:", len(df))
print("Explanation candidates:", len(explanation_df))

Original examples: 418545
Explanation candidates: 140020


In [51]:
def new_dataset_for_model(df):
    explanation_dataset = df[["INSTRUCTION", "RESPONSE"]]
    print("Total examples:", len(explanation_dataset))
    print(explanation_dataset.head())

new_dataset_for_model(explanation_df)

Total examples: 140020
                                          INSTRUCTION  \
0   Can you tell what is the following Python 3 fu...   
1   Can you generate a brief explanation for the f...   
12  Can you tell what is the following Python 3 fu...   
17  Can you generate a brief explanation for the f...   
18  Explain what the following Python 3 code does\...   

                                             RESPONSE  
0   Train a k - nearest neighbors classifier for f...  
1   Predicts faces in a given image using a KNN cl...  
12  Parses a string into a base - level structure ...  
17  Create a converter to drop the names of fields...  
18  Returns a verifier that checks the type of obj...  


In [52]:
def prepare_dataset(df):
    explanation_dataset = df[["INSTRUCTION", "RESPONSE"]]

    explanation_dataset["text"] = (
        "### Instruction:\n" + explanation_dataset["INSTRUCTION"] +
        "\n\n### Response:\n" + explanation_dataset["RESPONSE"]
    )

    print(explanation_dataset.head())
    print("Total examples:", len(explanation_dataset))

    return explanation_dataset

explanation_dataset = prepare_dataset(explanation_df)

                                          INSTRUCTION  \
0   Can you tell what is the following Python 3 fu...   
1   Can you generate a brief explanation for the f...   
12  Can you tell what is the following Python 3 fu...   
17  Can you generate a brief explanation for the f...   
18  Explain what the following Python 3 code does\...   

                                             RESPONSE  \
0   Train a k - nearest neighbors classifier for f...   
1   Predicts faces in a given image using a KNN cl...   
12  Parses a string into a base - level structure ...   
17  Create a converter to drop the names of fields...   
18  Returns a verifier that checks the type of obj...   

                                                 text  
0   ### Instruction:\nCan you tell what is the fol...  
1   ### Instruction:\nCan you generate a brief exp...  
12  ### Instruction:\nCan you tell what is the fol...  
17  ### Instruction:\nCan you generate a brief exp...  
18  ### Instruction:\nExplain what

In [53]:
from sklearn.model_selection import train_test_split

train_df, eval_df = train_test_split(explanation_dataset, test_size=0.1, random_state=42)

print("Training examples:", len(train_df))
print("Evaluation examples:", len(eval_df))

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
eval_dataset = Dataset.from_pandas(eval_df, preserve_index=False)

print("Training examples:", len(train_dataset))
print("Evaluation examples:", len(eval_dataset))
print("Columns:", train_dataset.column_names)

Training examples: 126018
Evaluation examples: 14002
Training examples: 126018
Evaluation examples: 14002
Columns: ['INSTRUCTION', 'RESPONSE', 'text']
